# Notebook 01: Clean and Join the Google Play Store Data

*Valvoline App Market Analysis · Part 1 of 2 · Python, pandas, ijson*

This notebook builds the analysis dataset used in **`02_analysis.ipynb`**.

**Inputs** (in Google Drive, `Colab Notebooks/`)
- `Google-Playstore.csv`: metadata for 2,312,944 Play Store apps (24 columns: category, rating, rating count, installs, size, price, ads, in-app purchases, …).
- `googleplay-app-permission.json`: the permissions each app requests, keyed by App Id.

**Steps**
1. Load the app metadata.
2. Join each app's permissions, streaming the large JSON file instead of loading it all into memory.
3. Remove apps with no rating or fewer than 10 ratings (2.31M → 988,310 apps), and add a log-scaled installs target.
4. Engineer features: app size in MB, monetization type, and permission-based features (sensitive access, permission load, privacy policy).

**Output:** `df_clean.parquet` (988,310 apps × 44 columns).

## 1. Setup and load the app metadata

In [ ]:
!pip install -q ijson

import ijson
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"
APPS_PATH = f"{DATA_DIR}/Google-Playstore.csv"
PERMISSIONS_PATH = f"{DATA_DIR}/googleplay-app-permission.json"
CLEAN_PATH = f"{DATA_DIR}/df_clean.parquet"

In [ ]:
df = pd.read_csv(APPS_PATH)
display(df.head())

,App Name,App Id,Category,Rating,Rating Count,Installs,Minimum Installs,Maximum Installs,Free,Price,...,Developer Website,Developer Email,Released,Last Updated,Content Rating,Privacy Policy,Ad Supported,In App Purchases,Editors Choice,Scraped Time
0,Gakondo,com.ishakwe.gakondo,Adventure,0.0,0.0,10+,10.0,15,True,0.0,...,https://beniyizibyose.tk/#/,jean21101999@gmail.com,"Feb 26, 2020","Feb 26, 2020",Everyone,https://beniyizibyose.tk/projects/,False,False,False,2021-06-15 20:19:35
1,Ampere Battery Info,com.webserveis.batteryinfo,Tools,4.4,64.0,"5,000+",5000.0,7662,True,0.0,...,https://webserveis.netlify.app/,webserveis@gmail.com,"May 21, 2020","May 06, 2021",Everyone,https://dev4phones.wordpress.com/licencia-de-uso/,True,False,False,2021-06-15 20:19:35
2,Vibook,com.doantiepvien.crm,Productivity,0.0,0.0,50+,50.0,58,True,0.0,...,NaN,vnacrewit@gmail.com,"Aug 9, 2019","Aug 19, 2019",Everyone,https://www.vietnamairlines.com/vn/en/terms-an...,False,False,False,2021-06-15 20:19:35
3,Smart City Trichy Public Service Vehicles 17UC...,cst.stJoseph.ug17ucs548,Communication,5.0,5.0,10+,10.0,19,True,0.0,...,http://www.climatesmarttech.com/,climatesmarttech2@gmail.com,"Sep 10, 2018","Oct 13, 2018",Everyone,NaN,True,False,False,2021-06-15 20:19:35
4,GROW.me,com.horodyski.grower,Tools,0.0,0.0,100+,100.0,478,True,0.0,...,http://www.horodyski.com.pl,rmilekhorodyski@gmail.com,"Feb 21, 2020","Nov 12, 2018",Everyone,http://www.horodyski.com.pl,False,False,False,2021-06-15 20:19:35


In [ ]:
# Display basic information about the DataFrame, including data types and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2312944 entries, 0 to 2312943
Data columns (total 24 columns):
 #   Column             Dtype  
---  ------             -----  
 0   App Name           object 
 1   App Id             object 
 2   Category           object 
 3   Rating             float64
 4   Rating Count       float64
 5   Installs           object 
 6   Minimum Installs   float64
 7   Maximum Installs   int64  
 8   Free               bool   
 9   Price              float64
 10  Currency           object 
 11  Size               object 
 12  Minimum Android    object 
 13  Developer Id       object 
 14  Developer Website  object 
 15  Developer Email    object 
 16  Released           object 
 17  Last Updated       object 
 18  Content Rating     object 
 19  Privacy Policy     object 
 20  Ad Supported       bool   
 21  In App Purchases   bool   
 22  Editors Choice     bool   
 23  Scraped Time       object 
dtypes: bool(4), float64(4), int64(1), object(15)
memor

`Installs` is a text bucket such as `"5,000+"`. Converting it to a number shows the format, but the analysis uses the numeric **Minimum Installs** column, which holds the same values.

In [ ]:
# Clean the 'Installs' column:
# 1. Remove '+' and ',' characters
# 2. Convert to numeric

df['Installs_cleaned'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Display the unique values in the original 'Installs' column to confirm cleaning approach was suitable
print("Unique values in original 'Installs' column before cleaning (first 10):\n", df['Installs'].unique()[:10])

# Display the first few rows with the new 'Installs_cleaned' column
print("\nDataFrame head with new 'Installs_cleaned' column:")
display(df[['App Name', 'Installs', 'Installs_cleaned']].head())

Unique values in original 'Installs' column before cleaning (first 10):
 ['10+' '5,000+' '50+' '100+' '1,000+' '500+' '50,000+' '10,000+' '1+'
 '500,000+']

DataFrame head with new 'Installs_cleaned' column:


,App Name,Installs,Installs_cleaned
0,Gakondo,10+,10.0
1,Ampere Battery Info,"5,000+",5000.0
2,Vibook,50+,50.0
3,Smart City Trichy Public Service Vehicles 17UC...,10+,10.0
4,GROW.me,100+,100.0


## 2. Join the permissions data
The permissions file is too large to load at once, so `ijson` streams it one record at a time. Only apps that exist in the metadata are kept. For each app it records:
- `permission_count`: the total number of permissions requested
- `permission_types`: the distinct permission groups (Camera, Location, …)
- `num_permission_types`: how many distinct groups that is

In [ ]:
app_ids = set(df["App Id"])
permission_rows = []

with open(PERMISSIONS_FILE, "r", encoding="utf-8") as f:
    for record in ijson.items(f, "item"):
        app_id = record.get("appId")

        if app_id not in app_ids:
            continue

        all_permissions = record.get("allPermissions", []) or []

        permission_types = sorted({
            p.get("type")
            for p in all_permissions
            if p.get("type")
        })

        permission_rows.append({
            "App Id": app_id,
            "permission_count": len(all_permissions),
            "permission_types": permission_types,
            "num_permission_types": len(permission_types)
        })

print(f"Permission records kept: {len(permission_rows):,}")

Permission records kept: 2,288,800


In [ ]:
permissions = pd.DataFrame(permission_rows)
print(f"Apps with permission data: {len(permissions):,}")
print(permissions.head())

Apps with permission data: 2,288,800
                                              App Id  permission_count  \
0             com.weddingstagedecorations.linastudio                 2   
1  com.bestappsstickers.app.stickersgif_lovegif_R...                 9   
2                  com.google.android.apps.messaging                32   
3                                       com.zoho.crm                29   
4                         tsc.multipurposemirror.com                 1   

                                    permission_types  num_permission_types  
0                                            [Other]                     1  
1               [Other, Photos/Media/Files, Storage]                     3  
2  [Camera, Contacts, Device ID & call informatio...                    11  
3  [Camera, Contacts, Device ID & call informatio...                    11  
4                                            [Other]                     1  


A **left join** on App Id keeps every app, including the few with no permission record. The row count must be identical before and after; a change would mean duplicate App Ids in the permissions data.

In [ ]:
# Run once: merging twice would create duplicate _x / _y columns
assert "permission_count" not in df.columns, "Already merged. Reload df and start over if this looks wrong."

rows_before = len(df)
df = df.merge(permissions, on="App Id", how="left")
print(f"Rows before: {rows_before:,} | after: {len(df):,}")   # Verify: must be identical

Rows before: 2,312,944 | after: 2,312,944


Apps with no permission record get zero permissions. The permission-type lists are stored as text (`"Camera; Location"`) so they save cleanly to parquet.

In [ ]:
# Run once: a second run would wipe the permission types
assert df["permission_count"].isna().any(), "Already ran this cell. Rerun the join cells, then this one."

# Apps without permission information get zeros
df["permission_count"] = df["permission_count"].fillna(0).astype(int)
df["num_permission_types"] = df["num_permission_types"].fillna(0).astype(int)

df["permission_types"] = df["permission_types"].apply(lambda x: x if isinstance(x, list) else [])

# Lists -> text ("Camera; Location"), split back apart by the flag code below
df["permission_types"] = df["permission_types"].apply(lambda x: "; ".join(x))

# Verify
print("Missing permission_count:", df["permission_count"].isna().sum())   # expect 0
print(df["permission_types"].head())                                       # expect text with semicolons

Missing permission_count: 0
0                                        Camera; Other
1                                                Other
2    Camera; Other; Phone; Photos/Media/Files; Storage
3                                                Other
4    Camera; Contacts; Device ID & call information...
Name: permission_types, dtype: object


One True/False column per sensitive permission group (`perm_Camera`, `perm_Location`, …), used as model features in notebook 02.

In [ ]:
FOCUS_TYPES = ['Camera', 'Location', 'Microphone', 'Contacts', 'SMS',
               'Phone', 'Storage', 'Photos/Media/Files', 'Calendar']

for t in FOCUS_TYPES:
    col = 'perm_' + t.split('/')[0].replace(' ', '_')   # e.g. perm_Camera, perm_Photos
    df[col] = df['permission_types'].astype(str).apply(
        lambda types_str: t in [p.strip() for p in types_str.split('; ') if p.strip()]
    )

perm_flag_cols = ['perm_' + t.split('/')[0].replace(' ', '_') for t in FOCUS_TYPES]
print(perm_flag_cols)
print(df[perm_flag_cols].sum())

['perm_Camera', 'perm_Location', 'perm_Microphone', 'perm_Contacts', 'perm_SMS', 'perm_Phone', 'perm_Storage', 'perm_Photos', 'perm_Calendar']
perm_Camera         547387
perm_Location       678246
perm_Microphone     248817
perm_Contacts       285599
perm_SMS              2971
perm_Phone          500465
perm_Storage       1383747
perm_Photos        1383877
perm_Calendar        91999
dtype: int64


## 3. Clean the joined data
- **Remove apps with no rating, or fewer than 10 ratings.** An average of one or two ratings is noise, and unrated apps can't be compared on rating at all. This cuts 2.31M apps to 988,310.
- **Add a log-scaled installs target.** Installs range from 0 to 10 billion. `log1p` (log of 1 + installs) compresses that range so a few giant apps don't dominate the models; the +1 handles apps with 0 installs.

In [ ]:
print(f"Duplicate App Ids: {df['App Id'].duplicated().sum():,}")
print(f"Starting rows: {len(df):,}")

# Remove missing / zero ratings
df = df.dropna(subset=["Rating"])
df = df[df["Rating"] > 0]
print(f"After removing unrated apps: {len(df):,}")

# Remove apps with fewer than 10 ratings
df = df[df["Rating Count"] >= 10]
print(f"After requiring 10+ ratings: {len(df):,}")

Duplicate App Ids: 0
Starting rows: 2,312,944
After removing unrated apps: 1,230,299
After requiring 10+ ratings: 988,310


In [ ]:
# Installs target: use the numeric Minimum Installs column (NOT the Installs string)
print("Missing Minimum Installs:", df["Minimum Installs"].isna().sum())

# log1p compresses the huge range (0 to billions). The +1 avoids an error on 0 installs.
df["log_installs"] = np.log1p(df["Minimum Installs"])

Missing Minimum Installs: 0


## 4. Engineer features
**4a. App size in MB.** Sizes are text like `"2.9M"`, `"512k"` or `"Varies with device"`. Parsed values are converted to MB, and anything unparseable becomes missing.

In [ ]:
print(df["Size"].unique()[:30])

['2.9M' '2.5M' '16M' '3.5M' '51M' '7.6M' '29M' '12M' '70M' '20M' '2.4M'
 '25M' '11M' '6.2M' '2.6M' '3.7M' '3.9M' '21M' '4.9M' '4.3M' '2.8M' '38M'
 '7.1M' '24M' '27M' '4.1M' '8.4M' '2.7M' '49M' '7.0M']


In [ ]:
def parse_size(s):
    s = str(s).replace(",", "").strip()
    try:
        if s.endswith("M"): return float(s[:-1])
        if s.endswith("k"): return float(s[:-1]) / 1024
        if s.endswith("G"): return float(s[:-1]) * 1024
    except ValueError:
        return np.nan
    return np.nan

df["Size_MB"] = df["Size"].apply(parse_size)
print(f"Missing Size_MB: {df['Size_MB'].isna().sum():,}")
print(df["Size_MB"].describe())
# Missing Size_MB (3.6% of apps) is filled with the median when building model features in notebook 02

Missing Size_MB: 35,144
count    953166.000000
mean         21.211326
std          26.284303
min           0.003125
25%           5.200000
50%          11.000000
75%          28.000000
max        1536.000000
Name: Size_MB, dtype: float64


**4b. Monetization type.** Each app gets one label: Paid, Free, Free+Ads, Free+IAP (in-app purchases) or Free+IAP+Ads. `np.select` takes the first matching condition, so the most specific combination comes first.

In [ ]:
conditions = [
    ~df["Free"],
    df["Free"] & df["Ad Supported"] & df["In App Purchases"],
    df["Free"] & df["In App Purchases"],
    df["Free"] & df["Ad Supported"],
]
choices = ["Paid", "Free+IAP+Ads", "Free+IAP", "Free+Ads"]
df["Monetization_Type"] = np.select(conditions, choices, default="Free")
print(df["Monetization_Type"].value_counts())

Monetization_Type
Free+Ads        480668
Free            348526
Free+IAP+Ads    105280
Free+IAP         32419
Paid             21417
Name: count, dtype: int64


**4c. Permission-based features**
- `Sensitive_Access`: requests camera, microphone or location
- `Permission_Load`: total permissions bucketed into Minimal / Moderate / Heavy / Very Heavy
- `Permission_Efficiency`: installs per permission requested
- `Has_Privacy_Policy`: the app lists a privacy policy

In [ ]:
# True if the app touches camera, microphone, or location
df["Sensitive_Access"] = df[["perm_Camera", "perm_Location", "perm_Microphone"]].any(axis=1)

# Buckets of total permissions requested
df["Permission_Load"] = pd.cut(df["permission_count"], bins=[-1, 5, 15, 30, 999],
                               labels=["Minimal", "Moderate", "Heavy", "Very Heavy"])

# Installs earned per permission requested (only defined when permission_count > 0)
df["Permission_Efficiency"] = np.where(df["permission_count"] > 0,
                                       df["Minimum Installs"] / df["permission_count"], np.nan)

# For the compliance-gap chart
df["Has_Privacy_Policy"] = df["Privacy Policy"].notna()

print(df["Sensitive_Access"].mean())
print(df["Permission_Load"].value_counts())

0.3656261699264401
Permission_Load
Moderate      545892
Minimal       344777
Heavy          95039
Very Heavy      2602
Name: count, dtype: int64


## 5. Save the clean dataset
Parquet keeps column types (booleans, categories) and loads much faster than CSV in notebook 02.

In [ ]:
df.to_parquet(CLEAN_PATH)
print("Saved:", df.shape)

Saved: (988310, 44)
